<a href="https://colab.research.google.com/github/sapkale12345/Research-Paper-Assistant-using-RAG/blob/main/RAG_Evaluation_pavan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# Install Required Libraries
# ==========================================

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip install -q \
ragas==0.2.15 \
langchain==0.3.25 \
langchain-community==0.3.24 \
langchain-core==0.3.63 \
langchain-chroma \
langchain-huggingface \
langchain-groq \
chromadb \
datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.0/363.0 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/7

In [ ]:
# ==========================================
# Verify Installation
# ==========================================

import ragas
import langchain
import chromadb
import langchain_groq

print("RAGAS      :", ragas.__version__)
print("LangChain  :", langchain.__version__)
print("ChromaDB   :", chromadb.__version__)

print("\nEnvironment Ready")

RAGAS      : 0.2.15
LangChain  : 0.3.25
ChromaDB   : 1.5.9

Environment Ready


In [ ]:
# ==========================================
# Import Libraries
# ==========================================

from google.colab import userdata

from datasets import Dataset

from langchain_chroma import Chroma

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate

from ragas import evaluate

from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall
)

from ragas.llms import LangchainLLMWrapper

from ragas.embeddings import LangchainEmbeddingsWrapper

In [ ]:
# ==========================================
# Load Embedding Model
# ==========================================

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vector_db = Chroma(
    persist_directory="/content/drive/MyDrive/Colab Notebooks/Research Paper/chroma_db",
    embedding_function=embedding_model
)

In [ ]:
# ==========================================
# Create Retriever
# ==========================================

retriever = vector_db.as_retriever(

    search_type="mmr",

    search_kwargs={

        "k":4,

        "fetch_k":10

    }

)

In [ ]:
# ==========================================
# Load Groq
# ==========================================

groq_api_key = userdata.get("GROQ_API_KEY")

llm = ChatGroq(

    model="llama-3.3-70b-versatile",

    api_key=groq_api_key,

    temperature=0

)

In [ ]:
# ==========================================
# Evaluation Dataset
# ==========================================

evaluation_data = [

    {
        "question": "what is Multi-Head Attention?",
        "expected_pages": [4,5]
    },

    {
        "question": "Why is self-attention?",
        "expected_pages": [6,7]
    },

    {
        "question": "Why is positional encoding required?",
        "expected_pages": [6]
    },

    {
        "question": "What datasets were used?",
        "expected_pages": [7]
    },

    {
        "question": "Applications of Attention in model?",
        "expected_pages": [5]
    }

]

In [ ]:
# ==========================================
# Retriever Metrics
# ==========================================

def recall_at_k(expected_pages, retrieved_pages):

    hits = len(set(expected_pages) & set(retrieved_pages))

    return hits / len(expected_pages)


def precision_at_k(expected_pages, retrieved_pages):

    hits = len(set(expected_pages) & set(retrieved_pages))

    return hits / len(retrieved_pages)


def hit_rate(expected_pages, retrieved_pages):

    return int(
        len(set(expected_pages) & set(retrieved_pages)) > 0
    )

In [ ]:
# ==========================================
# Evaluate Retriever
# ==========================================

results = []

for sample in evaluation_data:

    question = sample["question"]

    expected_pages = sample["expected_pages"]

    docs = retriever.invoke(question)

    retrieved_pages = sorted(

        set(

            doc.metadata["page"] + 1

            for doc in docs

        )

    )

    recall = recall_at_k(
        expected_pages,
        retrieved_pages
    )

    precision = precision_at_k(
        expected_pages,
        retrieved_pages
    )

    hit = hit_rate(
        expected_pages,
        retrieved_pages
    )

    results.append({

        "Question": question,

        "Expected Pages": expected_pages,

        "Retrieved Pages": retrieved_pages,

        "Recall@k": round(recall,2),

        "Precision@k": round(precision,2),

        "Hit Rate": hit

    })

In [ ]:
import pandas as pd

retriever_df = pd.DataFrame(results)

retriever_df

,Question,Expected Pages,Retrieved Pages,Recall@k,Precision@k,Hit Rate
0,what is Multi-Head Attention?,"[4, 5]","[3, 5, 13]",0.5,0.33,1
1,Why is self-attention?,"[6, 7]","[3, 5, 6, 13]",0.5,0.25,1
2,Why is positional encoding required?,[6],"[2, 5, 6]",1.0,0.33,1
3,What datasets were used?,[7],"[1, 7, 9, 12]",1.0,0.25,1
4,Applications of Attention in model?,[5],"[3, 5, 6, 13]",1.0,0.25,1


In [ ]:
print("="*60)
print("Retriever Evaluation")
print("="*60)

print(f"Average Recall@k     : {retriever_df['Recall@k'].mean():.2f}")

print(f"Average Precision@k  : {retriever_df['Precision@k'].mean():.2f}")

print(f"Average Hit Rate     : {retriever_df['Hit Rate'].mean():.2f}")

Retriever Evaluation
Average Recall@k     : 0.80
Average Precision@k  : 0.28
Average Hit Rate     : 1.00


In [ ]:
# ==========================================
# Create RAG Prompt
# ==========================================

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an AI Research Paper Assistant.

Answer ONLY using the retrieved context.

Instructions:
1. Use only the retrieved context.
2. Do not use outside knowledge.
3. If the answer is not available, reply:
   "I could not find this information in the research paper."
4. Keep the answer clear and concise.

Retrieved Context:
{context}
"""
        ),

        (
            "human",
            "{question}"
        )
    ]
)

In [ ]:
# ==========================================
# Create RAG Chain
# ==========================================

chain = prompt | llm

In [ ]:
questions = [
    "What is the main contribution of the paper?",
    "What is self-attention?",
    "Why is positional encoding required?",
    "What datasets were used?",
    "What evaluation metric was used?"
]

In [ ]:
ground_truths = [

    "The paper introduces the Transformer architecture based entirely on attention mechanisms.",

    "Self-attention allows each token to attend to every other token in the sequence.",

    "Positional encoding provides order information because the Transformer has no recurrence or convolution.",

    "The paper uses the WMT 2014 English-German and English-French translation datasets.",

    "BLEU score is used for evaluation."
]

In [ ]:
# ==========================================
# Generate Answers and Contexts (Using Subset)
# ==========================================

answers = []
contexts = []

for question in questions:

    docs = retriever.invoke(question)

    context = [doc.page_content for doc in docs]

    response = chain.invoke(
        {
            "context": "\n\n".join(context),
            "question": question
        }
    )

    answers.append(response.content)
    contexts.append(context)

In [ ]:
# ==========================================
# Create RAGAS Dataset (Using Subset)
# ==========================================

from datasets import Dataset

dataset = Dataset.from_dict(
    {
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths
    }
)

In [ ]:
# ==========================================
# Create RAGAS Wrappers
# ==========================================

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

ragas_llm = LangchainLLMWrapper(llm)

ragas_embeddings = LangchainEmbeddingsWrapper(
    embedding_model
)

In [ ]:
# ==========================================
# Define RAGAS Metrics
# ==========================================

from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall
)

faithfulness = Faithfulness(
    llm=ragas_llm
)

answer_relevancy = AnswerRelevancy(
    llm=ragas_llm,
    embeddings=ragas_embeddings
)

context_precision = ContextPrecision(
    llm=ragas_llm
)

context_recall = ContextRecall(
    llm=ragas_llm
)

In [ ]:
# ==========================================
# Run RAGAS Evaluation
# ==========================================

from ragas import evaluate

result = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ]
)

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[15]: TimeoutError()


In [ ]:
# ==========================================
# Display Results
# ==========================================

print("=" * 60)
print("Generation Evaluation")
print("=" * 60)

print(result)

Generation Evaluation
{'faithfulness': 0.9500, 'answer_relevancy': 0.7872, 'context_precision': 0.5944, 'context_recall': 1.0000}
